# DiT world state — where is position information, by residual point × diffusion time

**Question.** The DiT's representation of the frame being generated is indexed by TWO coordinates: the residual
point ℓ (depth) and the diffusion time τ of the prediction slot's iterate. Where in that (ℓ × τ) grid is the
world state (object positions) linearly decodable? This is the map the latent-steering experiment
(`../input_grad_steering/input_grad_steering_dit.ipynb`) selects its steering points from.

**Data / model provenance.** Model `9_dset4_dit_w4_d256` = **DiT concat · d256 · window 4 (4 ctx frames)**,
checkpoint `runs/dit/9_dset4_dit_w4_d256/best_model.pt` (best mean-mode val MSE 0.02445; row copied from
`DIT_RUNS.md`). Dataset `datasets/4_fixed_refl_inview` test split. Probes:
`pim.extractors.fit_readability_probes` (linear lstsq + 2×256 ReLU MLP, 80/20 by-sequence split).

## Definitions

| term | definition | units | better |
|---|---|---|---|
| **residual point ℓ** | the residual stream collected via the trunk's `resid_sink`: 0 = token embedding (input to block 1; NB it ingests `concat(obs[t], x_τ)`, so it is *not* the shared encoder port of the GRU/MSE-transformer), 1–3 = input to blocks 2–4, 4 = final-block output (what the velocity head reads) | — | — |
| **diffusion time τ (of the state)** | the prediction slot's noise level at the moment the state is read. States are collected from the **actual fresh-noise 8-step Euler sampler**: run the ODE to the Euler step where τ has the given value, read the residual stream with the current iterate in the slot. τ=1 = pure noise (context-only, the model's default `activations` view up to the noise draw); τ=0 = the finished sample | — | — |
| **linear / MLP probe R²** | held-out position R² (both objects, 4 dims) of the standard probes fit *at that (ℓ, τ) point* — one independent probe per grid cell | — | ↑ |

Context tokens are always clean (τ=0); only the prediction slot carries the iterate — matching both training's
`p_clean` pattern and deployment. States align "after obs[:, t]"; targets are `positions[:, t]` with the
all-objects-visible mask, exactly as in every other probe fit in the thread.

In [ ]:
# [1] Setup + collect residual states from the Euler sampler at every (residual point, τ) grid cell.
import os, sys
sys.path.insert(0, "../../../..")
import numpy as np, torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from pim.world_models.loader import load_checkpoint, load_dataset
from pim.extractors import fit_readability_probes
from pim.figures.theme import style_ax

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_OBJ, N_PROBE, CHUNK = 2, 500, 4096
OUT = "/tmp/dit_world_state"; os.makedirs(OUT, exist_ok=True)

MODEL_LABEL = "DiT concat · d256 · window 4"
model, info = load_checkpoint("../../../../runs/dit/9_dset4_dit_w4_d256/best_model.pt", device=DEVICE)
bundle = load_dataset("../../../../datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ, require_edits=False)
test = bundle.test
W, R, D = model.cfg.window, test.obs_res, model.cfg.d_model
S = model.cfg.n_sample_steps                       # 8 Euler steps
N_RESID = model.cfg.n_layers + 1
RESID_LABELS = ["0 · token embedding", "1 · early", "2 · middle", "3 · late", "4 · last (velocity-head input)"]
TAU_KS = [0, 2, 4, 6, 8]                           # Euler step indices → τ = 1.0, .75, .5, .25, 0.0
taus_full = torch.linspace(1.0, 0.0, S + 1)
TAU_VALS = [float(taus_full[k]) for k in TAU_KS]
print(f"model {MODEL_LABEL} | epoch {info.epoch} best mean-mode val MSE {info.val_loss:.5f} | "
      f"grid: {N_RESID} residual points × τ ∈ {TAU_VALS} | N_PROBE={N_PROBE} seqs | device={DEVICE}")

obs = torch.from_numpy(test.obs[:N_PROBE]).float().to(DEVICE)
B, T, _ = obs.shape
windows, lengths = model._unfold_windows(obs)                       # (B, T-1, W, R)
flat_win = windows.reshape(B * (T - 1), W, R)
flat_len = lengths.unsqueeze(0).expand(B, -1).reshape(-1)
n_rows = flat_win.shape[0]

# STATES[k][L] : (n_rows, D) — residual stream at point L with the Euler iterate at step k in the slot
STATES = {k: [np.empty((n_rows, D), np.float32) for _ in range(N_RESID)] for k in TAU_KS}
gen = torch.Generator().manual_seed(7)
eps_rows = torch.randn(n_rows, R, generator=gen)                    # fresh noise per window, fixed for the fit

with torch.no_grad():
    for i0 in range(0, n_rows, CHUNK):
        sl = slice(i0, min(i0 + CHUNK, n_rows))
        win, ln = flat_win[sl], flat_len[sl]
        n = win.shape[0]
        cur, nxt = model._window_tokens(win)
        attn = model._window_attn_mask(ln, DEVICE)
        x = eps_rows[sl].to(DEVICE)
        for k in range(S + 1):
            tau_t = torch.zeros(n, W, device=DEVICE); tau_t[:, -1] = taus_full[k]
            nxt_k = torch.cat([nxt[:, :-1], x.unsqueeze(1)], dim=1)
            if k in TAU_KS:
                sink = []
                feats, c = model._trunk(cur, nxt_k, tau_t, attn, resid_sink=sink)
                for L in range(N_RESID):
                    STATES[k][L][sl] = sink[L][:, -1].cpu().numpy()
                v = model.final_layer(feats, c)[:, -1]              # reuse the same forward for the ODE step
            else:
                v = model._denoise(cur, nxt_k, tau_t, attn)[:, -1]
            if k < S:
                x = x + (taus_full[k + 1] - taus_full[k]) * v
print(f"collected {n_rows:,} windows × {len(TAU_KS)} τ points × {N_RESID} residual points")

In [ ]:
# [2] Fit the standard probes at every grid cell; render the (residual point × τ) R² map.
Tm1 = T - 1
P_tgt = test.positions[:N_PROBE, :Tm1].reshape(N_PROBE, Tm1, N_OBJ * 2).astype(np.float32)
vis = test.is_visible[:N_PROBE, :Tm1, :N_OBJ].all(axis=2)

GRID = {}   # (L, k) -> fit dict
lin_r2 = np.zeros((N_RESID, len(TAU_KS)))
mlp_r2 = np.zeros((N_RESID, len(TAU_KS)))
for j, k in enumerate(TAU_KS):
    for L in range(N_RESID):
        states = STATES[k][L].reshape(N_PROBE, Tm1, D)
        fit = fit_readability_probes(states, P_tgt, mask=vis, device=DEVICE)
        GRID[(L, k)] = fit
        lin_r2[L, j], mlp_r2[L, j] = fit["linear_r2"], fit["mlp_r2"]

hdr = "| residual point | " + " | ".join(f"τ={t:g}" for t in TAU_VALS) + " |\n"
sep = "|---|" + "---|" * len(TAU_VALS) + "\n"
rows = "\n".join(f"| {RESID_LABELS[L]} | " + " | ".join(f"{lin_r2[L, j]:.3f}" for j in range(len(TAU_KS))) + " |"
                 for L in range(N_RESID))
display(Markdown("**Held-out LINEAR position R² by (residual point × τ of the iterate)** "
                 f"({GRID[(0, 0)]['n_train_seq']}/{GRID[(0, 0)]['n_heldout_seq']} train/held-out sequences)\n\n"
                 + hdr + sep + rows))

# Fig 1 — the grid, linear (a) and MLP (b) panels
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8))
for ax, mat, name in [(axes[0], lin_r2, "(a) linear probe"), (axes[1], mlp_r2, "(b) MLP probe")]:
    im = ax.imshow(mat, vmin=0, vmax=1, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(TAU_VALS)))
    ax.set_xticklabels([f"τ={t:g}" for t in TAU_VALS])
    ax.set_yticks(range(N_RESID)); ax.set_yticklabels(RESID_LABELS, fontsize=8)
    for L in range(N_RESID):
        for j in range(len(TAU_VALS)):
            ax.text(j, L, f"{mat[L, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if mat[L, j] < 0.6 else "black")
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("diffusion time of the prediction-slot iterate (1 = pure noise, 0 = finished sample)")
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle(f"Fig 1 — held-out position R² by residual point × diffusion time ({MODEL_LABEL}, "
             "states from the fresh-noise Euler sampler)", fontsize=11)
fig.tight_layout()
fig.savefig(f"{OUT}/fig1_probe_grid.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# [3] Sampled generative rollouts — is multi-step Euler sampling reliable autoregressively?
#     Fresh start-noise per rollout step (the 2026-08-11 fix; the model's fixed-bank "sample" mode collapses
#     into vertical stripes when iterated — see DIT_RUNS.md). Note: the fresh vector is shared across the
#     batch at each step (per-sample noise awaits the model-API fix); it is fresh ACROSS steps, which is what
#     stability needs.
from pim.world_models.dit import SingleFrameDiTModel  # noqa: F401  (loader dispatch)

model_sf, info_sf = load_checkpoint("../../../../runs/dit/11_dset4_dit_sf_w5_d256/best_model.pt", device=DEVICE)
SF_LABEL = "DiT single-frame · d256 · window 5"
EF, K_ROLL, N_EVAL = 20, 15, 64
obs_eval = torch.from_numpy(test.obs[:N_EVAL]).float().to(DEVICE)
clean_eval = test.clean_obs[:N_EVAL]
gt_body = clean_eval[:, EF:EF + K_ROLL]

def free_run(m, mode, seed=None):
    """Teacher-force noisy obs[0..EF-1], then K-step free-run in the given predict_mode.
    mode='sample' redraws the Euler start noise each step from `seed`."""
    m.predict_mode = mode
    gen = torch.Generator().manual_seed(seed) if seed is not None else None
    state = None
    with torch.no_grad():
        for t in range(EF):
            _, state = m.step(obs_eval[:, t], state)
        out = []
        for _ in range(K_ROLL):
            if mode == "sample" and gen is not None:
                m._eps_bank[0].copy_(torch.randn(m.cfg.input_dim, generator=gen).to(DEVICE))
            x = m.decode(state)
            out.append(x.cpu().numpy())
            _, state = m.step(x, state)
    # restore the deterministic bank + default mode
    g0 = torch.Generator().manual_seed(m.cfg.noise_seed)
    m._eps_bank.copy_(torch.randn(m.cfg.n_mean_eps, m.cfg.input_dim, generator=g0).to(DEVICE))
    m.predict_mode = "mean"
    return np.stack(out, axis=1)                     # (N, K, R)

ROLL_SETS = {}
for label, m in [(MODEL_LABEL, model), (SF_LABEL, model_sf)]:
    ROLL_SETS[label] = {"mean": free_run(m, "mean"),
                        "sample s0": free_run(m, "sample", seed=101),
                        "sample s1": free_run(m, "sample", seed=202)}

rows = []
for label, rolls in ROLL_SETS.items():
    for mode, r in rolls.items():
        rmse = float(np.sqrt(((r - gt_body) ** 2).mean()))
        rows.append(f"| {label} | {mode} | {rmse:.3f} |")
display(Markdown(f"**{K_ROLL}-step free-run RMSE vs clean GT** (N={N_EVAL} test sequences, free-run starts at "
                 f"frame {EF}; sampled rollouts intrinsically carry a reproduced observation-noise realisation "
                 "≈ the 0.2 obs-noise floor — compare the two sampled seeds to each other, and their "
                 "*structure* to GT, not their raw RMSE to mean mode)\n\n"
                 "| model | rollout mode | RMSE vs clean GT |\n|---|---|---|\n" + "\n".join(rows)))

In [ ]:
# [4] Figs 2–3 — sampled-rollout waterfalls, one figure per model.
#     Columns: GT | mean mode | Euler sample (2 independent noise seeds); each column its OWN free-run.
N_CTX = 6
BG, FG, TICK = "#0a0a14", "#a3adc2", "#808a9d"
SHOW = [0, 1]
ctx = test.obs[SHOW][:, EF - N_CTX:EF]

def rollout_waterfall(fig_no, label, rolls, fname):
    cols = [("GT (sim clean obs)", clean_eval[SHOW][:, EF:EF + K_ROLL]),
            ("Mean readout (deterministic)", rolls["mean"][SHOW]),
            ("Euler sample · fresh noise · seed A", rolls["sample s0"][SHOW]),
            ("Euler sample · fresh noise · seed B", rolls["sample s1"][SHOW])]
    fig, axes = plt.subplots(len(SHOW), len(cols), figsize=(3.1 * len(cols), 3.4 * len(SHOW)),
                             squeeze=False, facecolor=BG)
    for r, smp in enumerate(SHOW):
        for c, (name, body) in enumerate(cols):
            ax = axes[r][c]; ax.set_facecolor(BG)
            panel = np.clip(np.concatenate([ctx[r], body[r]], axis=0), 0, 1)
            ax.imshow(panel, cmap="gray", vmin=0, vmax=1, aspect="auto", interpolation="nearest")
            ax.axhline(N_CTX - 0.5, color="#fa8850", ls="--", lw=1.3)
            for sp in ax.spines.values(): sp.set_edgecolor(TICK)
            if r == 0: ax.set_title(name, fontsize=8.5, color=FG)
            if c == 0:
                ax.set_ylabel(f"test seq {smp}\nsim frame", fontsize=8, color=FG)
                ax.set_yticks([0, N_CTX, N_CTX + 7, N_CTX + 14])
                ax.set_yticklabels([EF - N_CTX, EF, EF + 7, EF + 14], fontsize=7)
            else: ax.set_yticks([])
            ax.set_xlabel("ray", fontsize=8, color=FG); ax.tick_params(colors=TICK, labelsize=7)
    fig.suptitle(f"Fig {fig_no} — autoregressive free-run by rollout mode ({label})\n"
                 f"{N_CTX} noisy teacher-forced context frames above the dashed line; below it every column "
                 "is its own free-run from step 0; GT column shows the clean render", fontsize=9.5, color=FG)
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    fig.savefig(fname, dpi=150, facecolor=BG, bbox_inches="tight"); plt.show()

rollout_waterfall(2, MODEL_LABEL, ROLL_SETS[MODEL_LABEL], f"{OUT}/fig2_sampled_rollouts_concat.png")
rollout_waterfall(3, SF_LABEL, ROLL_SETS[SF_LABEL], f"{OUT}/fig3_sampled_rollouts_single_frame.png")

## Current results (updated 2026-08-11; probe-standard fix applied same day)

**Probe grid (cell [2], Fig 1):**
- **Linear position R² is depth-monotone and essentially flat across τ**: token embedding 0.26 → late block
  0.70 → 0.76, with τ=1 / 0.75 / 0.5 / 0.25 within ±0.01 at every depth and a small +0.05 bump at τ=0. The
  world state in the generation-time representation is **context-driven**: fully present at pure noise; the
  iterate adds almost nothing until it is finished.
- **MLP probe (after the 2026-08-11 `STD_EPOCHS` 30→300 fix — the earlier negative values were an undertrained
  probe, see `pim/extractors/standard.py`): ≥ linear everywhere**, 0.84–0.89 at late/last blocks, same flat-in-τ
  shape, and **0.53–0.71 at the token embedding where linear reads only 0.26** — the pair-token embedding
  carries substantially more position information than is linearly accessible.
- **Best steering point: residual point 3 (late), any intermediate τ** (linear R² ≈ 0.70) — used by the
  latent-steering experiment.

**Sampled generative rollouts (cells [3]–[4], Figs 2–3):**
- **Fresh-noise Euler sampling is autoregressively stable on both d256 variants**: over 15 free-run steps both
  objects persist as coherent noisy bands, no collapse, no stripes, across independent seeds; seeds agree on
  structure and differ in noise realisation.
- RMSE vs clean GT: concat 0.227 (mean) vs 0.253/0.262 (sampled); single-frame 0.235 vs 0.271/0.273 — below the
  √(struct² + noise²) ≈ 0.30 a noise-floor-only account would predict, so sampled structure ≥ mean-mode
  structure.
- Residual limits (trajectory wobble, occasional near-merging) are shared with mean mode — model dynamics, not
  sampler pathology. The historical "sample mode collapses" observation is fully explained by the fixed
  start-noise bug (`DIT_RUNS.md`).

## Summary (interpretation — clearly marked as such)

The flat-in-τ map says the DiT's "belief" about where objects are lives in the **context pathway**, not in the
iterate: probing the generation midway reads the same information as probing before generation starts (and the
fixed MLP panel shows this is not a linearity artifact — the nonlinear readout is flat in τ too). For editing
that cuts both ways — a probe at intermediate τ is readable, but its information comes from the frozen clean
context tokens, so a gradient on x_τ alone must fight the fact that most of what the probe reads does not flow
through x_τ — which is what the latent-steering experiment found (duplication, not relocation). The
sampled-rollout section closes the reliability question: multi-step Euler generation is a trustworthy
generative mode once the start noise is redrawn per step.